In [5]:
# 1️⃣ First, upgrade huggingface_hub to a compatible version
!pip install -U "huggingface_hub>=0.28.0"

# 2️⃣ Reinstall PEFT to ensure it builds against the correct version
!pip install -U peft

# 3️⃣ (Optional) Reinstall transformers to align versions
!pip install -U transformers
!pip install -U torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

# 2️⃣ Ensure all Hugging Face core libs align
!pip install -U "transformers==4.45.2" "huggingface_hub>=0.28.0" "peft==0.13.2" "accelerate==0.34.2" "datasets==2.19.1"


  Using cached huggingface_hub-1.0.1-py3-none-any.whl.metadata (13 kB)
Using cached huggingface_hub-1.0.1-py3-none-any.whl (503 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.45.2 requires huggingface-hub<1.0,>=0.23.2, but you have huggingface-hub 1.0.1 which is incompatible.
tokenizers 0.20.3 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.0.1 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.0a1 which is incompatible.
  Using cached peft-0.17.1-py3-none-any.whl.metadata (14 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
Using cached peft-0.17.1-py3-none-any.w

## Restart Session after this

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import re
from peft import LoraConfig, get_peft_model, PeftModel
from safetensors.torch import load_file
import pandas as pd
import os
from pathlib import Path

2025-11-03 10:58:31.478532: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762167511.535576    1940 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762167511.553623    1940 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# Extraction Functions

In [7]:
# Sinhala detection
SINHALA_PATTERN = re.compile(r'[\u0D80-\u0DFF]')

# QA markers
Q_MARKERS = ['Q:', 'Q.', 'පු:']
A_MARKERS = ['A:', 'A.', 'උ:','c:','C:']
ALL_MARKERS = Q_MARKERS + A_MARKERS

def clean_marker(text):
    """
    Normalize QA markers:
    - 'පු:' and 'Q.', 'Q:' → 'Q:'
    - 'උ:' and 'A.', 'A:' → 'A:'
    """
    text = text.strip()

    for q_marker in ['පු:', 'Q.', 'Q:']:
        if text.startswith(q_marker):
            text = text[len(q_marker):].strip()
            return 'Q:' + text

    for a_marker in ['A:', 'A.', 'උ:','c:','C:']:
        if text.startswith(a_marker):
            text = text[len(a_marker):].strip()
            return 'A:' + text

    return text

def extract_sinhala_block(text, lookahead=10):
    """
    Extract Sinhala text block from given text.
    """
    match = re.search(SINHALA_PATTERN, text)
    if not match:
        return ""

    text = text[match.start():]
    result = []
    i = 0
    while i < len(text):
        result.append(text[i])
        if text[i] in ['.', '!', '?', '”', '"', '\n']:
            snippet = text[i+1:i+1+lookahead]
            if not re.search(SINHALA_PATTERN, snippet):
                break
        i += 1

    block = ''.join(result).rstrip()
    while block and not SINHALA_PATTERN.search(block[-1]):
        block = block[:-1]
    return block.strip()

def extract_qa_with_signatures(text, signature_map=None, sig_prefix="§SIG", start_counter=1):
    """
    Extract Sinhala QA blocks, replace them in text with signatures, and update signature map.

    Args:
        text (str): Original text with English + Sinhala QA
        signature_map (dict): Existing signature map (updated in-place)
        sig_prefix (str): Prefix for signatures
        start_counter (int): Starting signature number

    Returns:
        processed_text (str): Text with Sinhala replaced by signatures
        signature_map (dict): Updated mapping signature -> Sinhala
        next_sig_counter (int): Next counter value for further signatures
    """
    signature_map = signature_map or {}
    sig_counter = start_counter

    qa_pairs = []

    pattern = r'(' + '|'.join(re.escape(m) for m in ALL_MARKERS) + r')'
    markers = [(m.group(), m.start()) for m in re.finditer(pattern, text)]

    if not markers:
        return text, signature_map, sig_counter

    current_q_sig = None
    processed_text = text

    for i, (marker, pos) in enumerate(markers):
        end = markers[i+1][1] if i+1 < len(markers) else len(text)
        block = text[pos:end].strip()
        normalized_block = clean_marker(block)
        sinhala_text = extract_sinhala_block(normalized_block)

        if not sinhala_text:
            continue

        if normalized_block.startswith("Q:") and not sinhala_text.endswith("?"):
            sinhala_text += "?"
        elif normalized_block.startswith("A:") and not sinhala_text.endswith("."):
            sinhala_text += "."

        # Assign signature
        sig = f"{sig_prefix}{sig_counter:04d}"
        signature_map[sig] = sinhala_text
        sig_counter += 1

        # Replace Sinhala in normalized block with signature
        new_block = normalized_block.replace(sinhala_text, sig)
        processed_text = processed_text.replace(block, new_block, 1)

        # Track QA pairs using signatures
        if marker in Q_MARKERS:
            current_q_sig = sig
        elif marker in A_MARKERS and current_q_sig:
            qa_pairs.append({'Q': current_q_sig, 'A': sig})
            current_q_sig = None

    return processed_text, signature_map, sig_counter

# Translation Functions

In [8]:
def translate_sinhala_to_english(signature_map, max_tokens=1024):
    """
    Translates Sinhala contexts in a dictionary to English using NLLB-200 3.3B,
    with token limit awareness.

    Args:
        signature_map (dict): {signature: sinhala_text}
        max_tokens (int): maximum token length for each chunk (default 1024)
    Returns:
        dict: {signature: english_translation}
    """

    df = pd.read_csv("/kaggle/input/sin-eng/sin-eng.csv")
    legal_dict = dict(zip(df['sinhala'], df['english']))
    # Load base model
    adapter_path = "/kaggle/input/nllb-fine-tuned-600/kaggle/working/nllb_sinhala2english_lora"

    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device='cpu'
    base_model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/nllb-200-distilled-600M",
    torch_dtype=torch.float16 ,
    # device_map={"": device}
    device_map = 'auto'
    )
    # Load LoRA adapter
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj","v_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type="SEQ_2_SEQ_LM"
        )
    
    # 3️⃣ Wrap base model
    model = get_peft_model(base_model, lora_config)

    # 4️⃣ Load LoRA weights manually using safetensors
    adapter_weights = load_file(f"{adapter_path}/adapter_model.safetensors", device= 'cpu')#"cuda:0")
    model.load_state_dict(adapter_weights, strict=False)
    
    # 5️⃣ Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(adapter_path)

    model = model.to(device)
    model.eval()

    src_lang = "sin_Sinh"
    tgt_lang = "eng_Latn"
    
   
    translated_map = {}

    for sig, sinhala_text in signature_map.items():
        if legal_dict and sinhala_text in legal_dict:
            translated_map[sig] = legal_dict[sinhala_text]
            continue
        tokens = tokenizer(sinhala_text, return_tensors="pt", truncation=False).input_ids[0]
        token_len = len(tokens)

        # If the text fits within token limit, translate directly
        if token_len <= max_tokens:
            chunks = [sinhala_text]
        else:
            # Split text by sentences to fit token limit
            sentences = sinhala_text.split("।")
            chunks = []
            current_chunk = ""

            for sentence in sentences:
                candidate = (current_chunk + " " + sentence).strip()
                candidate_tokens = tokenizer(candidate, return_tensors="pt").input_ids[0]

                if len(candidate_tokens) > max_tokens:
                    chunks.append(current_chunk.strip())
                    current_chunk = sentence
                else:
                    current_chunk = candidate

            if current_chunk:
                chunks.append(current_chunk.strip())

        # Translate each chunk and combine
        translated_chunks = []
        for chunk in chunks:
            if legal_dict:
                for sin_phrase, eng_translation in legal_dict.items():
                    if sin_phrase in chunk:
                        chunk = chunk.replace(sin_phrase, eng_translation)

            inputs = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True).to(device)
            translated_tokens = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang)
            )
            english_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
            translated_chunks.append(english_text)
        translated_map[sig] = " ".join(translated_chunks).strip()


    return translated_map

def replace_signatures_with_translations(translated_text, translated_map):
    """
    Replaces signature placeholders in a translated text with their English translations.

    Args:
        translated_text (str): Text containing signatures like §SIG0001
        translated_map (dict): {signature: english_translation}

    Returns:
        str: Text with signatures replaced by their translated context
    """

    # Regex to match §SIGxxxx patterns
    pattern = r"§SIG\d{4}"

    def replacer(match):
        sig = match.group(0)
        return translated_map.get(sig, sig)

    return re.sub(pattern, replacer, translated_text)



In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

def translate_sinhala_to_english(signature_map, legal_dict=None, max_tokens=1024):
    """
    Translates Sinhala texts in a dictionary to English using LoRA-adapted NLLB-200 600M,
    with dictionary fallback, token-limit-aware chunking, and safe GPU inference.

    Args:
        signature_map (dict): {signature: sinhala_text}
        legal_dict (dict, optional): {sinhala_phrase: english_translation} to prioritize
        max_tokens (int): maximum token length per chunk
    Returns:
        dict: {signature: english_translation}
    """

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load model + tokenizer
    base_model_name = "facebook/nllb-200-distilled-600M"
    adapter_dir = "/kaggle/working/nllb_sinhala2english_lora"

    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)
    model = PeftModel.from_pretrained(base_model, adapter_dir, is_trainable=False)
    model.to(device)
    model.eval()

    tgt_lang = "eng_Latn"  # forced target language
    translated_map = {}

    for sig, sinhala_text in signature_map.items():

        # 1️⃣ Full-text dictionary lookup
        if legal_dict and sinhala_text in legal_dict:
            translated_map[sig] = legal_dict[sinhala_text]
            continue

        # 2️⃣ Tokenize and check length
        tokens = tokenizer(sinhala_text, return_tensors="pt", truncation=False).input_ids[0]
        if len(tokens) <= max_tokens:
            chunks = [sinhala_text]
        else:
            # Split by sentences
            sentences = sinhala_text.split("।")
            chunks = []
            current_chunk = ""

            for sentence in sentences:
                candidate = (current_chunk + " " + sentence).strip()
                candidate_tokens = tokenizer(candidate, return_tensors="pt").input_ids[0]

                if len(candidate_tokens) > max_tokens:
                    # Current chunk is ready to append
                    if current_chunk:
                        chunks.append(current_chunk.strip())

                    # Check if sentence itself is too long
                    sentence_tokens = tokenizer(sentence, return_tensors="pt").input_ids[0]
                    if len(sentence_tokens) > max_tokens:
                        # Split sentence into sub-chunks of max_tokens
                        for i in range(0, len(sentence_tokens), max_tokens):
                            sub_tokens = sentence_tokens[i:i+max_tokens]
                            sub_chunk = tokenizer.decode(sub_tokens, skip_special_tokens=True)
                            chunks.append(sub_chunk.strip())
                        current_chunk = ""
                    else:
                        current_chunk = sentence
                else:
                    current_chunk = candidate

            if current_chunk:
                chunks.append(current_chunk.strip())

        # 3️⃣ Translate each chunk with dictionary replacement
        translated_chunks = []
        for chunk in chunks:
            if legal_dict:
                for sin_phrase, eng_translation in legal_dict.items():
                    if sin_phrase in chunk:
                        chunk = chunk.replace(sin_phrase, eng_translation)

            # Model inference
            with torch.no_grad():
                inputs = tokenizer(
                    chunk,
                    return_tensors="pt",
                    truncation=True,
                    padding=True,
                    max_length=max_tokens
                ).to(device)
                outputs = model.generate(
                    **inputs,
                    forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
                    max_length=256
                )
                english_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
                translated_chunks.append(english_text)

            torch.cuda.empty_cache()  # prevent memory buildup

        # 4️⃣ Combine all chunks
        translated_map[sig] = " ".join(translated_chunks).strip()

    return translated_map


In [ ]:

text = """

As per the submitted facts of the case, the deceased Neela Malani Wakwella was the legally married wife of the Appellant, and according to the evidence of the mother of the Appellant, Kumarihamy (PW2), who was called as a prosecution witness at the High Court trial, the deceased and the Appellant lived in a house in close proximity to the house of Kumarihamy. The Police had commenced the investigation into the sudden death of the deceased Neela Malani Wakwella as a case of suicide by strangulation, but later, her husband (the Appellant) was arrested as the suspect for committing murder of the said deceased. The entirety of the prosecution’s case is based on circumstantial evidence placed before the High Court; therefore, it is important to conduct a proper evaluation of the said circumstantial evidence in order to address the first question of law submitted in this present case. During the trial before the High Court, the prosecution had relied on the evidence of the witnesses namely Godella Waththa Arachchilage Ranjith Dharmasiri (PW 1), Jayasinghe Mudiyanselage Kumarihamy (PW 2), Weerasinghe Mudiyanselage Podi Appuhamy (PW 6), Doctor Ashoka Bandara Senevirathne Consultant Judicial Medical Officer Kandy (PW 16), Retired Inspector of Police Marabedde Rathnayake Tiyunis Gunathilake (PW 9) and Widyarathne Ganithayalage Wasantha Kumari Premaratne (PW 8). According to the evidence of Kumarihamy (PW 2), who is the mother of the Appellant, the deceased was living alone with the Appellant, on the day the incident occurred. As narrated by PW 2, at around 8.30 pm on the day of the incident, the Appellant had visited her on his way to the paddy field to borrow a torch from her. She further testified that upon hearing the cries of the Appellant around 11.30 pm following his return from the paddy field, she had hastily made her way to the Appellant’s house to inquire. At this juncture, PW1 had seen a piece of wire hooked onto the beam of the house and another piece of wire that encircled the deceased’s neck. She further testified that she held the deceased by her legs, and when she attempted to bring her down, the wire SC APPEAL 16/2020 JUDGEMENT Page 5 of 15

from which the deceased was hanging was broken. During this time, the Appellant was outside the house crying for help from the neighbours. According to the evidence of witness Dharmasiri (PW 1- the Grama Seva Niladhari), he had gone to the house of the deceased on the day in question around midnight on information received from a neighbour. PW 1 found the Appellant seated on a mat outside the house, and discovered the deceased collapsed near a chair, with one of her legs still propped on the said chair. He also observed a piece of wire tied to the roof. He was informed that the deceased had committed suicide by hanging herself. He then had taken steps to the Police about the death by telephone. The above descriptions of the scene were confirmed by the evidence of the Inspector of Police Gunathilake, who was attached to Walapane Police Station. According to his evidence, the information with regard to the suicide of Wakwella Liyana Arachchige Neela Malani Wakwella was received by the police station around 2.40 A.M. from the Grama Niladhari Dharmasiri, and he had gone to the scene of crime around 3.10 A.M. with a team of police officers consisting of PC 40442, PC 15890, RPC 37718 and the Inquirer into Sudden Deaths (hereinafter sometimes referred to as the “ISP”) of the area. Whilst confirming the position of the dead body with the Grama Niladhari, this witness had further observed a piece of wire near the body of the deceased in addition to the wire hanging from the roof. This witness had further observed a wallet with several letters kept on a cabinet closer to the body. Since the witness had observed some significance with the said wallet, he had inspected it and found 27 well-packed love letters written by one Kumari to the Appellant, Roshan Bandaranayake. He had taken the said letters into his custody, and the said letters were identified at the trial before the High Court. According to the witness, the wallet was empty save for the letters. Even though he had arrived at the scene with the ISP, he felt suspicious of the nature of the wires and the letters found inside the wallet. Thereby, he requested the inquirer to refer the matter to a Magisterial Inquiry. SC APPEAL 16/2020 JUDGEMENT Page 6 of 15

The prosecution had relied on the evidence of Wasantha Kumari Premarathne (PW 8) to explain the 27 love letters found in close proximity to the dead body of the deceased. According to the evidence of Wasantha Kumari, she had an affair with the Appellant, Roshan Bandaranayake, in or about 2003 and admitted to writing letters to him. PW 8 had identified the letters produced before the court by the prosecution. However, according to this witness, she was unaware of the fact that the Appellant was a married person when she engaged in the affair. When the wife of the accused (the deceased in this case) visited her along with her mother to confront the witness, on her father's advice PW 8 worshipped the deceased to apologise and express her regret, promising to end the affair with the Appellant. According to the witness, since then, she has never written or maintained any relationship with the Appellant, and the letters she received from the Appellant were burnt by her. As it was submitted, the most important evidence of the prosecution was the evidence of the Consultant Judicial Medical Officer A.B. Seneviratne (PW 16), who held the post- mortem inquiry of the deceased Neela Malani Wakwella. At the post-mortem, Dr. Senevirathne stated that a total of 21 wounds were discovered on the body of the deceased, and there were multiple abrasions on both sides of the neck of the deceased. The Judicial Medical Officer (hereinafter referred to as 'the JMO'), in giving evidence at the trial, confirmed that the death was caused by manual strangulation. පු: ගෙලෙහි තිබූ බාහිර තුවාල සම්බන්ධයෙන් අභාාන්තර තුවාල සම්බන්ධයෙන් නිරීක්ෂණය කළේ මොනවාද? උ: ගෙලෙහි ඉදිරිපිට ඇති මාංශපේශි වලට ඇති වූ රුධිර වහනය නිසා ඇති වූ තුවාලය. ඉදිරිපස ඇති මාංශපේශි වල ඉහළ භාගයේ මෙම තැළීම් දක්නට ලැබුණා. එමෙන්ම රුධිර ගැලීම් වටා වැඩිපුර දක්නට ලැබුණා. ගෙලෙහි දකුණු පැත්තේ එමෙන්ම ඉදිරිපස ගැඹුරින් පිහිටා තිබූ මාංශපේශි වල තැළීම් හේතුකොට ගෙන තැලීම් තුවාල තිබුණා. තයිරොයිඩ් වල වම් පැත්තේ ඉහලට තෙරා ඇති කොටස බිදී තිබුණා. පු: ඒ ආකාරයෙන් තුවාළ සිදුවිය හැක්කේ කවර ආකාරයෙන්ද? උ: ස්වාමීනී, 1-9 දක්වා මා සඳහන් කර තිබෙනවා, ගෙලෙහි වම් පැත්තේ සහ දකුණු පැත්තේ යටි හනුවේ කෝණයට පහළින් පිහිටා තිබූ සීරීම් තුවාළ සම්බන්ධයෙන්. ඊට Page <b>7</b> of <b>15</b> <b>SC APPEAL 16/2020</b> <b>JUDGEMENT</b>

අමතරව ඊට යටින් ඇති මාංශ පේශි වල තැලීම් සහ උගුරු දණ්ඩෙහි වම් පස බග්නයක් මා සඳහන් කළා. ඒ ආකාරයට තුවාළ සිදු වී තිබුණේ ගෙල හිර කිරීමෙනාා. පු: ඔබතුමාගේ පශ්චාත් මරණ පරීක්ෂණ වාර්තාවේ මරණයට හේතුව ලෙස හඳුනාගෙන ඇත්තේ මොකක්ද? උ: අතින් ගෙල සිර කිරීමක්. In his evidence, the JMO categorically rejected the contention that this death was a suicide. පු: මෙය එල්ලීමෙන් සිදුකර ගනු ලැබූ සිය දිවි නසා ගැනීමක් ලෙස ඔබට සාක්ෂි හමු උතා ද? උ: නැත. පු: ඔබතුමාගේ ඒ මතයට පදනම් වූ හේතුවක් තිබෙනවාද? උ: එසේය. පු: ඒ මොකක්ද? උ: මා ඉහතින් සඳහන් කළ ආකාරයට ඇයගේ ගෙලෙහි සිරීම් තුවාල දක්නට ලැබුණා. ඉදිරිපස සිට පැත්තට වන්නට. ගෙලෙහි වම් පැත්තට වන්නට සිරීම් තුවාල වැඩි ප‰මාණයක් දක්නට ලැබුණා. එමෙන්ම මෙම සිරීම් තුවාල විශාල ප‰මාණයේ සිරීම් තුවාල නෙමෙයි. 1 වන තුවාලය සෙන්ටි මීටර් 3.52 යි. අනෙක් හැම එකක්ම කුඩා සීරීම් තුවාල. සමහර ඒවා සිරස් අතට සහ සිරසට ආනතව පිහිටා තිබුණා. එමෙන්ම එම සීරීම් තුවාල වලට පිටින් ඇති මාංශ පේශි කැපීමක් දක්නට ලැබුණා. උගුරු දණ්ඩේ භග්නයක් දක්නට ලැබුනා. එවැනි සීරීම් තුවාල ගෙල වැළලා ගෙන මිය යන අයගේ දක්නට ලැබෙන්නේ නැහැ. During cross-examination, the JMO stated that, පු: මම නැවත වරක් යෝජනා කරනවා. ඇයම ගෙල වැළලා ගෙන මෙම මරණය සිදු වුණේ කියලා? උ: නමුත් ගෙල වැළලා ගැනීමේදී දක්නට ලැබෙන ලක්ෂණ කිසිවක් මෙම මෘත ශරීරයේ දක්නට ලැබුණේ නැහැ. <b>SC APPEAL 16/2020</b> Page <b>8</b> of <b>15</b> <b>JUDGEMENT</b>

Further, the JMO specifically rejected all suggestions to the effect that the wounds discovered on deceased's body were self-inflicted as a result of sudden change of mind subsequent to the deceased's decision to hang herself. In cross-examination, පු: මේ සීරීම් තුවාල ගෙල ලා ගන්න කොට ඇති වෙන්නේ නැද්ද දගලන අවස්ථාවේදි ඇය විසින්ම මේ සීරීම් තුවාල කර ගන්න හැකියාවක් නැද්ද ගෙල ලා ගෙන මැරෙන්න හදන අවස්ථාවක්. ගෙල වැල ලා ගන්න අවස්ථාවක මරණය ඉතා ඉක්මනින් සිදු වෙනවා. c: බොහෝ අවස්ථාවල ඒ පුද්ගලයා අවසන් වශයෙන් ඔහුගේ තීරණය වෙනස් කලහොත් ඔහුටම බැරි වෙනවා ඔහුගේ ජීවිතය බේර ගන්න. එවැනි අවස්ථාවක ඕනී නම් තොණ්ඩුව දා ගන්න අවස්ථාවේදී ඇති වෙන්න පුලුවන්. මේ තරම් විශාල තැලීම් තුවාල ඇති වෙන්නේ නැහැ. මේ සීරීම් තුවාල විසිරී ඇති ආකාරයට සිරස් අතට සහ සිරසට ආතතව. පු: එල්ලිලා පහතට ඇදෙන අවස්ථාවක සීරීම් තුවාල ඇති වීමට හැකියාවක් නැද්ද? උ: මේ ආකාරයට ඇති වෙන්නේ නැහැ. Further, he specifically rejected all the suggestions to the effect that the wounds discovered on the body were caused due to falling as a result of the breaking of wires. පු: එම තැනැත්තිය, මිය ගිය තැනැත්තිය බිමට වැටුණා නම් වෛදාාතුමා කියන සීරීම් තුවාල ඇති විය හැකියි නේද? උ: නැත. පු: උඩකින් බිමට වැටෙන කොට මේ දණහිසට තුවාල සිදු වන්නේ නැද්ද? උ: විය හැකියි, නමුත් මා නිරීක්ෂණය කළේ මියගිය අයෙකුගේ සිරුරේ තියෙන තුවාල නෙමෙයි. අංක <math>10</math> වශයෙන් මම පැහැදිලිව ලකුණක් කළා. පු: මෙය මියගිය තැනැත්තිය උඩක සිට බිමට වැටීමෙන් සිදු විය හැකිද? උ: පුද්ගලයෙක් ඉහල තැනක සිට පහලට වැටුනාම සීරීම් තුවාල ඇති වෙන්න පුළුවන්. ඊට වැඩිය තැලීම් තුවාල වැඩියි. වැළ මිට, වළලුක දණහිසේ තුවාලවල එවැනි තැලීම් දක්නට ලැබුණේ නැහැ. කලවයේ පමණක් තැලීම් තුවාල මා සඳහන් කරලා තියෙනවා. එය හරස් අතට පිහිටා තිබූ තැලීම් තුවාලයක්. Page <b>9</b> of <b>15</b> <b>SC APPEAL 16/2020</b> JUDGEMENT


whose whereabouts were unknown for seven years (“ නත් වසරකින් ආගිය අතක් නොදත් පද්මිනී පෙරේරාගේ විවනක පුරුෂයා”). Section 52 of the Marriage Registration Ordinance


had drawn in stating මෙම ඉඩම තුල පදිංචිව සිටින අනික් එකම පාර්ශවය වූදිත පාර්ශවයයි. විනාඩි 5 ක් වන කෙටි කාලයකදී දියණියට පහරදීමක් සිදු කිරීමට reasonably draw an inference that the three offences were
"""

processed_text, signature_map, sig_counter = extract_qa_with_signatures(text)
print(signature_map)
# processed_text, signature_map, sig_counter = extract_quoted_bracket_sinhala_with_signatures(text=processed_text, signature_map=signature_map, start_counter=sig_counter)


text = processed_text
i = 0
n = len(text)
while i < n:
    if text[i] == '§':  # skip existing SIG
        end_sig = text.find(' ', i)
        if end_sig == -1:
            end_sig = n
        i = end_sig
        continue
    if SINHALA_PATTERN.match(text[i]):
        # Extract a block starting from here
        remaining_text = text[i:]
        block = extract_sinhala_block(remaining_text)
        if block:
            sig = f"§SIG{sig_counter:04d}"
            signature_map[sig] = block
            text = text[:i] + sig + text[i+len(block):]
            sig_counter += 1
            i += len(sig)  # skip past newly inserted sig
            n = len(text)
            continue
    i += 1



translated_map = translate_sinhala_to_english(signature_map)
final_translation = replace_signatures_with_translations(text, translated_map)

print(translated_map)

In [ ]:
print(processed_text)

## Process Directory

In [9]:
# Sinhala character detection pattern
SINHALA_PATTERN = re.compile(r'[\u0D80-\u0DFF]')

def process_text_file(input_path, output_dir):
    """
    Process a Sinhala+English text file, extract all Sinhala content,
    replace with signatures, translate Sinhala parts, and save results.
    """
    
    with open(input_path, "r", encoding="utf-8") as f:
        text = f.read()

    print(f"\n📄 Processing file: {os.path.basename(input_path)}")
    signature_map = {}
    sig_counter = 1

    # QA extractor
    processed_text, signature_map, sig_counter = extract_qa_with_signatures(text)

    # Remaining Sinhala blocks
    text = processed_text
    i = 0
    n = len(text)
    while i < n:
        if text[i] == '§':
            end_sig = text.find(' ', i)
            if end_sig == -1:
                end_sig = n
            i = end_sig
            continue
        if SINHALA_PATTERN.match(text[i]):
            remaining_text = text[i:]
            block = extract_sinhala_block(remaining_text)
            if block:
                sig = f"§SIG{sig_counter:04d}"
                signature_map[sig] = block
                text = text[:i] + sig + text[i+len(block):]
                sig_counter += 1
                i += len(sig)
                n = len(text)
                continue
        i += 1

    processed_text = text

    # Step 3️⃣ — Translate Sinhala map to English
    translated_map = translate_sinhala_to_english(signature_map)

    # Step 4️⃣ — Replace all signatures with English translations
    final_translation = replace_signatures_with_translations(processed_text, translated_map)

    base_name = os.path.splitext(os.path.basename(input_path))[0]
    translated_output = os.path.join(output_dir, f"{base_name}_translated.txt")

    with open(translated_output, "w", encoding="utf-8") as f:
        f.write(final_translation)

    print(f"✅ Translated file saved → {translated_output}")
    print(f"🧭 Total Sinhala blocks: {len(signature_map)}")

    return signature_map

def process_directory(input_dir, output_dir, recursive=True):
    """
    Process all text files in a directory through the Sinhala translation pipeline.

    Args:
        input_dir (str): Path to input directory containing .txt files
        output_dir (str): Path to output directory where results will be saved
        recursive (bool): If True, process subdirectories as well
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    files = list(input_dir.rglob("*.txt")) if recursive else list(input_dir.glob("*.txt"))

    count = 0
    skipped = 0
    for file_path in files:
        # Compute relative path and target output directory
        rel_path = file_path.relative_to(input_dir).parent
        target_output_dir = output_dir / rel_path
        os.makedirs(target_output_dir, exist_ok=True)

        # Compute translated output file path
        base_name = file_path.stem
        translated_output_file = target_output_dir / f"{base_name}_translated.txt"

        # Skip if already exists
        if translated_output_file.exists():
            print(f"⏭️ Skipping already translated file: {file_path.name}")
            skipped += 1
            continue

        # Otherwise, try processing
        try:
            process_text_file(file_path, target_output_dir)
            count += 1
        except Exception as e:
            print(f"⚠️ Error processing {file_path.name}: {e}")

    print(f"\n✅ Completed processing {count} file(s) from {input_dir}")
    if skipped > 0:
        print(f"⏭️ Skipped {skipped} file(s) that were already translated.")


input_dir = "/kaggle/input/cleaned-texts/CLEANED TEXTS/Civil_Appeal"
output_dir = "/kaggle/working/Translated_Civil"
os.makedirs(output_dir, exist_ok = True)
process_directory(input_dir, output_dir)

⏭️ Skipping already translated file: SC_CA_139_2017.txt
⏭️ Skipping already translated file: SC_CA_404_2013.txt
⏭️ Skipping already translated file: SC_CA_14_2014.txt
⏭️ Skipping already translated file: SC_CA_55_2010.txt
⏭️ Skipping already translated file: SC_CA_43_2019.txt
⏭️ Skipping already translated file: SC_CA_60_2018.txt
⏭️ Skipping already translated file: SC_CA_115_2013.txt
⏭️ Skipping already translated file: SC_CA_119_2023.txt
⏭️ Skipping already translated file: SC_CA_139_2019.txt
⏭️ Skipping already translated file: SC_CA_136_2014.txt
⏭️ Skipping already translated file: SC_CA_27_2019.txt
⏭️ Skipping already translated file: SC_CA_112_2011.txt
⏭️ Skipping already translated file: SC_CA_99_2009.txt
⏭️ Skipping already translated file: SC_CA_68_2015.txt
⏭️ Skipping already translated file: SC_CA_226_2014.txt
⏭️ Skipping already translated file: SC_CA_192_2014.txt
⏭️ Skipping already translated file: SC_CA_41_2015.txt
⏭️ Skipping already translated file: SC_CA_170_2011.txt


In [ ]:
!zip -r "/kaggle/working/Translated_Civil.zip" "/kaggle/working/Translated_Civil"

updating: kaggle/working/Translated_Civil/ (stored 0%)
updating: kaggle/working/Translated_Civil/SC_CA_154_2016_translated.txt (deflated 72%)
updating: kaggle/working/Translated_Civil/SC_CA_50_2018_translated.txt (deflated 69%)
updating: kaggle/working/Translated_Civil/SC_CA_24_2020_translated.txt (deflated 70%)
updating: kaggle/working/Translated_Civil/SC_CA_110_2018_translated.txt (deflated 70%)
updating: kaggle/working/Translated_Civil/SC_CA_147_2017_translated.txt (deflated 69%)
updating: kaggle/working/Translated_Civil/SC_CA_199_2012_translated.txt (deflated 66%)
updating: kaggle/working/Translated_Civil/SC_CA_35_2021_translated.txt (deflated 64%)
updating: kaggle/working/Translated_Civil/SC_CA_172_2011_translated.txt (deflated 66%)
updating: kaggle/working/Translated_Civil/SC_CA_190_2016_translated.txt (deflated 69%)
updating: kaggle/working/Translated_Civil/SC_CA_42_2010_translated.txt (deflated 68%)
updating: kaggle/working/Translated_Civil/SC_CA_194_2016_translated.txt (deflat

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 72%)
updating: kaggle/working/Translated_Civil/SC_CA_219_2016_translated.txt (deflated 63%)
updating: kaggle/working/Translated_Civil/SC_CA_113_2011_translated.txt (deflated 66%)
updating: kaggle/working/Translated_Civil/SC_CA_49_2016_translated.txt (deflated 65%)
updating: kaggle/working/Translated_Civil/SC_CA_125_2015_translated.txt (deflated 65%)
updating: kaggle/working/Translated_Civil/SC_CA_163_2015_translated.txt (deflated 69%)
updating: kaggle/working/Translated_Civil/SC_CA_197_2011_translated.txt (deflated 67%)
updating: kaggle/working/Translated_Civil/SC_CA_48_2013_1_translated.txt (deflated 69%)
updating: kaggle/working/Translated_Civil/SC_CA_114_2017_translated.txt (deflated 62%)
updating: kaggle/working/Translated_Civil/SC_CA_35_2008_translated.txt (deflated 68%)
updating: kaggle/working/Translated_Civil/SC_CA_02_2015_translated.txt (deflated 71%)
updating: kaggle/working/Translated_Civil/SC_CA_88_2021_translated.txt (deflated 67%)
updating: kaggle/working/Trans

In [13]:
import json
import os
import shutil

# Paths
WORKING_DIR = "/kaggle/working"
OUTPUT_DIR = os.path.join(WORKING_DIR, "translated")
ZIP_PATH = os.path.join(WORKING_DIR, "Translated_Civil.zip")

# Make sure the output folder exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Move the zip into the output folder
shutil.copy(ZIP_PATH, OUTPUT_DIR)

# Dataset metadata
metadata = {
    "title": "Translated-Civil2",
    "id": "MuaadhNazly/Translated-Civil2",
    "licenses": [{"name": "CC0-1.0"}]
}

# Save metadata JSON inside the output folder
metadata_path = os.path.join(OUTPUT_DIR, "dataset-metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f)

# Install/upgrade Kaggle API
!pip install kaggle --upgrade

# Set up Kaggle API key
os.makedirs("/root/.config/kaggle", exist_ok=True)
!cp "/kaggle/input/api-key-2/kaggle.json" "/root/.config/kaggle/"
!chmod 600 /root/.config/kaggle/kaggle.json

# Create the Kaggle dataset
!kaggle datasets create -p {OUTPUT_DIR} -u --quiet


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Your public Dataset is being created. Please check progress at https://www.kaggle.com/datasets/muaadhnazly/Translated-Civil2
